# TDT4173 – Submission 1

Group 71  
Students: Marco Josefsen, Theresa Kieu-Mi Tran, Bilal Saher
Kaggle team name: [71] Three Blind Mice

Score: 0.99574

Model was trained on a T4 GPU

In [ ]:
# -----------------------------
# Config
# -----------------------------
from pathlib import Path

RAW_DIR = Path(".")
OUTPUT_DIR = RAW_DIR
OUT_NAME = "submission_1.csv"
DEVICE = "cuda"
SEEDS = [1337, 2024, 7]
LP_WORKERS = None  # All available CPU cores.
PL_ROUNDS = 977

BASE_PARAMS = dict(
    objective="binary:logistic",
    eval_metric="auc",
    tree_method="hist",
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    min_child_weight=20,
    subsample=0.7,
    colsample_bytree=0.8,
    reg_lambda=20.0,
    reg_alpha=0.5,
    gamma=0.1,
    max_bin=256,
    n_jobs=4,
)


In [ ]:
# -----------------------------
# Imports and input files
# -----------------------------
import gc
import io
import json
import os
import re
import subprocess
import time
import warnings

import numpy as np
import pandas as pd
import scipy
import xgboost as xgb
import yaml
from IPython.display import display
from threadpoolctl import threadpool_limits

warnings.filterwarnings("ignore", category=FutureWarning)
if xgb.__version__ != "3.4.1":
    raise RuntimeError("Install XGBoost 3.4.1 before running this notebook.")
if DEVICE != "cpu" and not DEVICE.startswith("cuda"):
    raise ValueError("DEVICE must be 'cpu' or 'cuda'.")

RAW_DIR = RAW_DIR.expanduser().resolve()
OUTPUT_DIR = OUTPUT_DIR.expanduser().resolve()
SP = str(RAW_DIR)
WV_FILE = RAW_DIR / "data/kernel/Synthetic_water_value_2015_2024.csv"
REQUIRED_FILES = [
    "sample_submission.csv",
    "prediction_mapping.csv",
    "data/kernel/Historical_day_ahead_price_2015_2025.csv",
    "data/kernel/Historical_inflow_1958_2025.csv",
    "data/kernel/Historical_volume_2015_2024.csv",
    "data/kernel/Synthetic_water_value_2015_2024.csv",
    "data/kernel/Unit_commitment_decisions.csv",
    "data/extended/Constraint_min_flow.csv",
    "data/extended/Constraint_min_volume.csv",
    "data/extended/Tokke_Vinje_topology.yaml",
]
missing = [name for name in REQUIRED_FILES if not (RAW_DIR / name).is_file()]
if missing:
    raise FileNotFoundError(f"Check RAW_DIR. Missing input files: {missing}")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
T_START = time.time()
print("Data:", RAW_DIR)
print("XGBoost:", xgb.__version__, "| NumPy:", np.__version__, "| pandas:", pd.__version__)


In [ ]:
# -----------------------------
# Preprocessing helpers
# -----------------------------
import io, re

H = 168
TZ_LOCAL = "Europe/Oslo"
USE_END_OF_WEEK_VOLUME = False


def read_ts(path):
    df = pd.read_csv(path)
    df["date"] = pd.to_datetime(df["date"], utc=True)
    return df.set_index("date").sort_index()


def to_hourly_grid(df, start, end):
    hourly = df.resample("h").mean()
    grid = pd.date_range(start, end, freq="h", tz="UTC")
    return hourly.reindex(grid).interpolate("time", limit_direction="both")


def drop_constant(df):
    return df.drop(columns=[c for c in df.columns if df[c].nunique() <= 1])


def dedupe_columns(df):
    keep = []
    for c in df.columns:
        if next((k for k in keep if df[k].equals(df[c])), None) is None:
            keep.append(c)
    return df[keep]


def windows(series_or_df, starts, lookback=0):
    """(n_cases, H + lookback, n_cols) hourly values for each case window [start − lookback, start + H)."""
    df = series_or_df.to_frame() if isinstance(series_or_df, pd.Series) else series_or_df
    pos = df.index.get_indexer(starts)
    assert (pos >= lookback).all() and (
        pos + H - 1 < len(df)
    ).all(), "case window outside the hourly grid"
    idx = pos[:, None] + np.arange(-lookback, H)[None, :]
    return df.to_numpy(dtype=np.float32)[idx]


def _csv_roundtrip(df, float_format):
    buf = io.StringIO()
    df.to_csv(buf, index=False, float_format=float_format)
    buf.seek(0)
    return pd.read_csv(buf)


In [ ]:
# -----------------------------
# Load and preprocess the raw data
# -----------------------------
def preprocess(SP):
    KERNEL, EXTENDED = f"{SP}/data/kernel", f"{SP}/data/extended"
    price_raw = read_ts(f"{KERNEL}/Historical_day_ahead_price_2015_2025.csv")
    inflow_raw = read_ts(f"{KERNEL}/Historical_inflow_1958_2025.csv")
    volume_raw = read_ts(f"{KERNEL}/Historical_volume_2015_2024.csv")
    wv_raw = read_ts(f"{KERNEL}/Synthetic_water_value_2015_2024.csv")
    minvol_raw = read_ts(f"{EXTENDED}/Constraint_min_volume.csv")
    minflow_raw = read_ts(f"{EXTENDED}/Constraint_min_flow.csv")
    uc = pd.read_csv(f"{KERNEL}/Unit_commitment_decisions.csv")
    test_map = pd.read_csv(f"{SP}/prediction_mapping.csv")
    sample_sub = pd.read_csv(f"{SP}/sample_submission.csv")
    topo = yaml.safe_load(open(f"{EXTENDED}/Tokke_Vinje_topology.yaml", encoding="utf-8"))

    # Put the series on a complete hourly UTC grid.
    GRID_START, GRID_END = "2014-12-01", "2025-01-08 23:00"
    price = to_hourly_grid(price_raw, GRID_START, GRID_END)["Day-ahead price (EUR/MWh)"].rename(
        "price"
    )
    inflow = drop_constant(to_hourly_grid(inflow_raw, GRID_START, GRID_END))
    minvol = drop_constant(to_hourly_grid(minvol_raw, "2015-01-01", GRID_END))
    minflow = drop_constant(to_hourly_grid(minflow_raw, "2015-01-01", GRID_END))
    volume = drop_constant(volume_raw)
    wv = dedupe_columns(wv_raw)

    # Map plants to their connected reservoirs.
    model = topo["model"]
    max_vol = {r: v["max_vol"] for r, v in model["reservoir"].items()}
    conns = pd.DataFrame(topo["connections"])

    def follow_to_reservoir(node, direction):
        key, other, other_type = (
            ("to", "from", "from_type") if direction == "up" else ("from", "to", "to_type")
        )
        frontier, seen, found = [node], set(), []
        while frontier:
            n = frontier.pop()
            for _, c in conns[conns[key] == n].iterrows():
                if c[other_type] == "reservoir":
                    found.append(c[other])
                elif c[other_type] in ("tunnel", "river") and c[other] not in seen:
                    seen.add(c[other])
                    frontier.append(c[other])
        return found

    plants = list(model["plant"].keys())
    plant_map = {
        p: {"up": follow_to_reservoir(p, "up"), "down": follow_to_reservoir(p, "down")}
        for p in plants
    }
    target_cols = [c for c in sample_sub.columns if c != "Run No"]
    UNITS = list(
        dict.fromkeys(re.match(r"result_committed_(.+)_t\d+$", c).group(1) for c in target_cols)
    )
    unit_plant = {u: u.rsplit("_", 1)[0] for u in UNITS}

    def plant_wv(day_index):
        out = {}
        for p, m in plant_map.items():
            out[f"wv_up_{p}"] = wv_raw.loc[day_index, m["up"]].mean(axis=1).to_numpy()
            out[f"wv_down_{p}"] = (
                wv_raw.loc[day_index, m["down"]].mean(axis=1).to_numpy()
                if m["down"]
                else np.zeros(len(day_index))
            )
        return out

    cases = pd.concat(
        [
            uc[["Run No", "starttime"]].assign(split="train"),
            test_map[["Run No", "starttime"]].assign(split="test"),
        ],
        ignore_index=True,
    )
    cases["starttime"] = pd.to_datetime(cases["starttime"], utc=True)
    cases = cases.sort_values("Run No").reset_index(drop=True)
    assert (
        cases["Run No"].is_unique
        and cases["starttime"].diff().dropna().eq(pd.Timedelta("1D")).all()
    )
    Y_train = (
        uc.set_index("Run No")[target_cols]
        .to_numpy(dtype=np.float32)
        .reshape(len(uc), len(UNITS), H)
    )

    starts = pd.DatetimeIndex(cases["starttime"])
    N = len(cases)
    P = windows(price, starts)[..., 0]
    P_prev = windows(price, starts, lookback=H)[:, :H, 0]
    INF = windows(inflow, starts)
    MINV = windows(minvol, starts) if minvol.shape[1] else np.zeros((N, H, 0), np.float32)
    MINF = windows(minflow, starts) if minflow.shape[1] else np.zeros((N, H, 0), np.float32)

    # Case-level features.
    day0 = starts.normalize()
    day_end = day0 + pd.Timedelta("7D")
    f = {"Run No": cases["Run No"].to_numpy()}
    f["year"] = starts.year
    f["month"] = starts.month
    doy = starts.dayofyear.to_numpy()
    f["doy_sin"], f["doy_cos"] = np.sin(2 * np.pi * doy / 365.25), np.cos(2 * np.pi * doy / 365.25)
    f["price_mean"], f["price_std"] = P.mean(1), P.std(1)
    f["price_min"], f["price_max"] = P.min(1), P.max(1)
    f["price_q10"], f["price_q90"] = np.quantile(P, 0.1, axis=1), np.quantile(P, 0.9, axis=1)
    f["price_prev_week_mean"] = P_prev.mean(1)
    f["price_week_over_week"] = f["price_mean"] - f["price_prev_week_mean"]
    f["price_daily_amplitude"] = (P.reshape(N, 7, 24).max(2) - P.reshape(N, 7, 24).min(2)).mean(1)
    for r in volume.columns:
        v0 = volume.loc[day0, r].to_numpy()
        f[f"vol_{r}"] = v0
        f[f"fill_{r}"] = v0 / max_vol[r]
        if USE_END_OF_WEEK_VOLUME:
            f[f"dvol_{r}"] = volume.loc[day_end, r].to_numpy() - v0
    f["fill_total"] = sum(volume.loc[day0, r].to_numpy() for r in volume.columns) / sum(
        max_vol[r] for r in volume.columns
    )
    for r in wv.columns:
        f[f"wv0_{r}"] = wv.loc[day0, r].to_numpy()
        f[f"wv7_{r}"] = wv.loc[day_end, r].to_numpy()
        f[f"dwv_{r}"] = f[f"wv7_{r}"] - f[f"wv0_{r}"]
    pw_end = plant_wv(day_end)
    for p in plants:
        up, down = pw_end[f"wv_up_{p}"], pw_end[f"wv_down_{p}"]
        f[f"wvup_{p}"], f[f"wvnet_{p}"] = up, up - down
        f[f"spread_{p}"] = f["price_mean"] - up
        f[f"frac_price_gt_wv_{p}"] = (P > up[:, None]).mean(1)
    clim_src = inflow_raw.loc[:"2014-12-31", inflow.columns]
    clim = clim_src.groupby(clim_src.index.dayofyear).mean()
    inf_week = INF.mean(1)
    clim_days = ((doy[:, None] - 1 + np.arange(7)[None, :]) % 365) + 1
    clim_week = clim.to_numpy()[clim_days - 1].mean(1)
    for j, r in enumerate(inflow.columns):
        f[f"inflow_{r}"] = inf_week[:, j]
    f["inflow_total"] = inf_week.sum(1)
    f["inflow_total_anomaly"] = inf_week.sum(1) / (clim_week.sum(1) + 1e-6)
    for j, r in enumerate(minflow.columns):
        f[f"minflow_{r}"] = MINF[:, :, j].mean(1)
    for j, r in enumerate(minvol.columns):
        f[f"minvol_{r}"] = MINV[:, :, j].mean(1)
    case_feat = pd.DataFrame(f)
    case_feat.insert(1, "starttime", cases["starttime"].to_numpy())
    case_feat.insert(2, "split", cases["split"].to_numpy())
    case_feat["fold"] = np.where(case_feat["split"] == "train", case_feat["year"], -1)

    # Hourly features.
    t = np.arange(H)
    ts = starts.tz_convert(None).to_numpy()[:, None] + (t * np.timedelta64(1, "h"))[None, :]
    ts_local = pd.DatetimeIndex(ts.ravel()).tz_localize("UTC").tz_convert(TZ_LOCAL)
    hf = {
        "Run No": np.repeat(cases["Run No"].to_numpy(), H),
        "t": np.tile(t, N),
        "day_in_horizon": np.tile(t // 24, N),
        "hour_local": ts_local.hour.to_numpy(),
        "dow_local": ts_local.dayofweek.to_numpy(),
        "is_weekend": (ts_local.dayofweek >= 5).astype(np.int8),
    }
    hs = {}
    hs["price"] = P
    hs["price_minus_mean"] = P - P.mean(1, keepdims=True)
    hs["price_z"] = hs["price_minus_mean"] / (P.std(1, keepdims=True) + 1e-6)

    hs["price_rank_week"] = P.argsort(1, kind="stable").argsort(1, kind="stable") / (H - 1)
    Pd = P.reshape(N, 7, 24)
    hs["price_rank_day"] = (Pd.argsort(2, kind="stable").argsort(2, kind="stable") / 23).reshape(
        N, H
    )
    hs["price_minus_daymean"] = (Pd - Pd.mean(2, keepdims=True)).reshape(N, H)
    hs["price_diff_prev"] = np.diff(P, axis=1, prepend=P_prev[:, -1:])
    hs["price_diff_next"] = np.concatenate(
        [P[:, 1:] - P[:, :-1], np.zeros((N, 1), np.float32)], axis=1
    )
    k = 6
    pad = np.pad(P, ((0, 0), (k, k)), mode="edge")
    hs["price_roll13"] = np.stack([pad[:, i : i + H] for i in range(2 * k + 1)]).mean(0)
    for p in plants:
        hs[f"price_minus_wvup_{p}"] = P - pw_end[f"wv_up_{p}"][:, None]
    hs["inflow_total_h"] = INF.sum(2)
    for j, r in enumerate(minflow.columns):
        hs[f"minflow_h_{r}"] = MINF[:, :, j]
    for j, r in enumerate(minvol.columns):
        hs[f"minvol_h_{r}"] = MINV[:, :, j]
    for name, arr in hs.items():
        hf[name] = arr.astype(np.float32).ravel()
    hourly = pd.DataFrame(hf)
    Y_all = np.full((N, len(UNITS), H), np.nan, dtype=np.float32)
    train_mask = (cases["split"] == "train").to_numpy()
    assert np.array_equal(cases.loc[train_mask, "Run No"].to_numpy(), uc["Run No"].to_numpy())
    Y_all[train_mask] = Y_train
    for u_i, u in enumerate(UNITS):
        hourly[f"y_{u}"] = Y_all[:, u_i, :].ravel()

    hourly_feature_cols = list(hs.keys()) + [
        "t",
        "day_in_horizon",
        "hour_local",
        "dow_local",
        "is_weekend",
    ]
    static_feature_cols = [
        c for c in case_feat.columns if c not in ("Run No", "starttime", "split", "fold")
    ]
    y_cols = [f"y_{u}" for u in UNITS]
    meta = {
        "units": UNITS,
        "unit_plant": unit_plant,
        "plant_map": plant_map,
        "submission_columns": sample_sub.columns.tolist(),
        "hourly_feature_cols": hourly_feature_cols,
        "static_feature_cols": static_feature_cols,
        "y_cols": y_cols,
    }

    # Retain the precision used in the original preprocessing CSVs.
    hourly_cols = ["Run No"] + hourly_feature_cols
    is_train_row = np.repeat(train_mask, H)
    cases_s1 = _csv_roundtrip(case_feat, "%.6g")
    h_tr = _csv_roundtrip(
        hourly.loc[is_train_row, hourly_cols + y_cols].astype({c: "int8" for c in y_cols}), "%.6g"
    )
    h_te = _csv_roundtrip(hourly.loc[~is_train_row, hourly_cols], "%.6g")
    return cases_s1, h_tr, h_te, meta


In [ ]:
# -----------------------------
# Scheduling features
# -----------------------------
def window_mean(a, width, forward):
    a = np.asarray(a, dtype=np.float64)
    h = a.shape[1]
    t = np.arange(h)
    left = t if forward else np.maximum(0, t - width + 1)
    right = np.minimum(h, t + width) if forward else t + 1
    cumulative = np.pad(np.cumsum(a, axis=1), ((0, 0), (1, 0)))
    return (cumulative[:, right] - cumulative[:, left]) / (right - left)


def remaining_sum(a):
    return np.cumsum(np.asarray(a, dtype=np.float64)[:, ::-1], axis=1)[:, ::-1]


def block_total(values, active):
    left = np.zeros_like(values, dtype=float)
    right = np.zeros_like(left)
    for t in range(values.shape[1]):
        previous = left[:, t - 1] if t else 0
        left[:, t] = np.where(active[:, t], previous + values[:, t], 0)
    for t in range(values.shape[1] - 1, -1, -1):
        following = right[:, t + 1] if t + 1 < values.shape[1] else 0
        right[:, t] = np.where(active[:, t], following + values[:, t], 0)
    return left + right - np.where(active, values, 0)


def engineer_cases(source_cases, metadata):
    result = source_cases.drop(columns=[c for c in source_cases if c.startswith("dvol_")]).copy()
    new = {}
    for plant, mapping in metadata["plant_map"].items():
        side_fill = {}
        for side in ("up", "down"):
            columns = [f"fill_{r}" for r in mapping[side]]
            if columns and all(c in result for c in columns):
                side_fill[side] = result[columns].mean(axis=1).to_numpy()
                new[f"fe_initial_fill_{side}_{plant}"] = side_fill[side]
        if "up" in side_fill and "down" in side_fill:
            new[f"fe_initial_fill_up_minus_down_{plant}"] = side_fill["up"] - side_fill["down"]
    for column in source_cases:
        if not column.startswith("vol_"):
            continue
        reservoir = column[len("vol_") :]
        inflow_column = f"inflow_{reservoir}"
        if inflow_column not in result:
            continue
        initial = result[column].to_numpy(dtype=float)
        incoming = result[inflow_column].to_numpy(dtype=float) * H * 3600 / 1e6
        denominator = initial + incoming
        new[f"fe_inflow_share_of_initial_plus_week_{reservoir}"] = np.divide(
            incoming, denominator, out=np.zeros_like(incoming), where=denominator > 0
        )
    additions = pd.DataFrame(new, index=result.index, dtype=np.float32)
    return pd.concat([result, additions], axis=1), list(additions.columns)


def engineer_hourly(source_hourly, source_cases, metadata):
    result = source_hourly.drop(columns=[c for c in source_hourly if c.startswith("dvol_")])
    result = result.sort_values(["Run No", "t"]).reset_index(drop=True)
    ids = result["Run No"].drop_duplicates().to_numpy()
    assert result.groupby("Run No").size().eq(H).all() and np.array_equal(
        result["t"].to_numpy(), np.tile(np.arange(H), len(ids))
    )
    aligned = source_cases.set_index("Run No").loc[ids]
    shape = (len(ids), H)
    price = result["price"].to_numpy(dtype=float).reshape(shape)
    count_remaining = H - np.arange(H)
    new = {}
    for width in (6, 24):
        for direction, forward in (("past", False), ("next", True)):
            new[f"fe_price_minus_{direction}{width}h_mean"] = price - window_mean(
                price, width, forward
            )
    new["fe_remaining_price_max_minus_current"] = (
        np.maximum.accumulate(price[:, ::-1], axis=1)[:, ::-1] - price
    )
    new["fe_remaining_price_mean_minus_current"] = remaining_sum(price) / count_remaining - price
    extra = {}
    for plant in metadata["plant_map"]:
        spread = price - aligned[f"wvup_{plant}"].to_numpy(dtype=float)[:, None]
        above = spread > 0
        new[f"fe_price_minus_wvnet_{plant}"] = (
            price - aligned[f"wvnet_{plant}"].to_numpy(dtype=float)[:, None]
        )
        new[f"fe_above_wv_block_hours_{plant}"] = block_total(np.ones_like(spread), above)
        new[f"fe_remaining_above_wv_fraction_{plant}"] = remaining_sum(above) / count_remaining
        new[f"fe_remaining_positive_spread_mean_{plant}"] = (
            remaining_sum(np.maximum(spread, 0)) / count_remaining
        )
        water_value = aligned[f"wvup_{plant}"].to_numpy(dtype=float)[:, None]
        extra[f"fe_relative_price_spread_{plant}"] = spread / (
            np.abs(price) + np.abs(water_value) + 1.0
        )
    inflow = result["inflow_total_h"].to_numpy(dtype=float).reshape(shape)
    volume = inflow * 3600 / 1e6
    new["fe_inflow_before_hour_mm3"] = np.pad(np.cumsum(volume, axis=1)[:, :-1], ((0, 0), (1, 0)))
    new["fe_inflow_remaining_mm3"] = remaining_sum(volume)
    new["fe_inflow_minus_week_mean"] = inflow - inflow.mean(axis=1, keepdims=True)
    for column in result:
        if column.startswith("minvol_h_"):
            reservoir = column[len("minvol_h_") :]
            initial = aligned[f"vol_{reservoir}"].to_numpy(dtype=float)[:, None]
            new[f"fe_initial_volume_minus_minvol_{reservoir}"] = initial - result[column].to_numpy(
                dtype=float
            ).reshape(shape)
    new.update(extra)
    additions = pd.DataFrame({name: a.ravel().astype(np.float32) for name, a in new.items()})
    return pd.concat([result, additions], axis=1), list(additions.columns)


In [ ]:
# -----------------------------
# Build the scheduling tables
# -----------------------------
def build_control_tables(SP):
    cases_s1, h_tr_s1, h_te_s1, meta = preprocess(SP)
    cases_fe, added_static = engineer_cases(cases_s1, meta)
    h_tr, added_hourly = engineer_hourly(h_tr_s1, cases_s1, meta)
    h_te, added_te = engineer_hourly(h_te_s1, cases_s1, meta)
    assert added_te == added_hourly
    meta = dict(meta)
    meta["hourly_feature_cols"] = meta["hourly_feature_cols"] + added_hourly
    meta["static_feature_cols"] = meta["static_feature_cols"] + added_static
    return cases_fe, h_tr, h_te, meta


t0 = time.time()
cases, h_tr, h_te, meta = build_control_tables(SP)
print(f"control tables built from raw data in {time.time() - t0:.0f}s")


In [ ]:
# -----------------------------
# Assemble hourly model inputs
# -----------------------------
UNITS, HF, SF = meta["units"], meta["hourly_feature_cols"], meta["static_feature_cols"]
UNIT_PLANT = meta["unit_plant"]
PLANTS = list(dict.fromkeys(UNIT_PLANT[u] for u in UNITS))
YC = [f"y_{u}" for u in UNITS]
CONTROL = HF + SF
print(f"control features: {len(HF)} hourly + {len(SF)} static = {len(CONTROL)}")

cases["starttime"] = pd.to_datetime(cases["starttime"], utc=True)
cases = cases.sort_values("Run No").reset_index(drop=True)
h_te = h_te.reindex(columns=h_tr.columns)
hourly = (
    pd.concat([h_tr, h_te], ignore_index=True).sort_values(["Run No", "t"]).reset_index(drop=True)
)
del h_tr, h_te

N = len(cases)
assert (
    len(hourly) == N * 168
    and (hourly["Run No"].to_numpy().reshape(N, 168)[:, 0] == cases["Run No"].to_numpy()).all()
)
assert (
    hourly["t"].to_numpy().reshape(N, 168) == np.arange(168)
).all(), "hourly rows are not complete 0..167 cases"
assert hourly[CONTROL[: len(HF)]].notna().all().all() and cases[SF].notna().all().all()

LAB = (cases["split"] == "train").to_numpy()
print(f"cases: {N} ({LAB.sum()} labelled, {(~LAB).sum()} test) | hourly rows: {len(hourly):,}")
print(
    "labelled starts",
    cases.starttime[LAB].min().date(),
    "→",
    cases.starttime[LAB].max().date(),
    "| test starts",
    cases.starttime[~LAB].min().date(),
    "→",
    cases.starttime[~LAB].max().date(),
)

RUN = hourly["Run No"].to_numpy()
TT = hourly["t"].to_numpy()
Y = hourly[YC].to_numpy(np.float32)
FEAT = pd.DataFrame(
    np.hstack(
        [hourly[HF].to_numpy(np.float32), np.repeat(cases[SF].to_numpy(np.float32), 168, axis=0)]
    ),
    columns=CONTROL,
)
PRICE = hourly["price"].to_numpy(np.float64).reshape(N, 168)
MF = {
    c: hourly[c].to_numpy(np.float64).reshape(N, 168)
    for c in ["minflow_h_r_Vest_Vassdraget", "minflow_h_b_Vinjevatn", "minflow_h_b_Byrtevatn"]
}
del hourly


In [ ]:
# -----------------------------
# Topology and hourly inflow
# -----------------------------
topo = yaml.safe_load(open(f"{SP}/data/extended/Tokke_Vinje_topology.yaml"))
MAXVOL = {r: v["max_vol"] for r, v in topo["model"]["reservoir"].items()}
GEN = {}
for g, v in topo["model"]["generator"].items():
    curves = (
        v["turb_eff_curves"] if isinstance(v["turb_eff_curves"], list) else [v["turb_eff_curves"]]
    )
    GEN[g] = dict(
        p_min=v["p_min"],
        p_max=v["p_max"],
        sc=v["startcost_const"] + v["stopcost_const"],
        q_min=min(min(c["x"]) for c in curves),
        q_max=max(max(c["x"]) for c in curves),
    )
PLANT_QMAX = {p: sum(GEN[u]["q_max"] for u in UNITS if UNIT_PLANT[u] == p) for p in PLANTS}
print(pd.DataFrame(GEN).T)

infl = pd.read_csv(f"{SP}/data/kernel/Historical_inflow_1958_2025.csv")
infl["date"] = pd.to_datetime(infl["date"], utc=True)
infl = infl.set_index("date").sort_index()
assert infl.index.is_unique
st = cases["starttime"].dt.tz_convert(None).to_numpy()
ts = pd.DatetimeIndex((st[:, None] + np.arange(168) * np.timedelta64(1, "h")).ravel()).tz_localize(
    "UTC"
)
Q = infl.reindex(ts)
assert Q.notna().all().all(), "raw inflow missing for some case hours"
Q = {c: Q[c].to_numpy(np.float64).reshape(N, 168) for c in Q.columns}
del infl


def vol0(res):
    return (
        cases[f"vol_{res}"].to_numpy(np.float64)
        if f"vol_{res}" in cases
        else np.full(N, 0.5 * MAXVOL[res])
    )


In [ ]:
# -----------------------------
# Water-balance features
# -----------------------------
POOLS = {
    "Staavatn": (["Staavatn"], [], []),
    "Kjelavatn": (["Kjelavatn"], [], ["Staavatn"]),
    "Langesae": (["Langesae"], [], []),
    "Foersvatn+Bordalsvatn": (["Foersvatn", "Bordalsvatn"], [], ["Langesae"]),
    "Hyljelihyl": (["Hyljelihyl"], [], ["Kjelavatn", "Foersvatn+Bordalsvatn"]),
    "Songavatn+Bitdalsvatn": (["Songavatn", "Bitdalsvatn"], [], []),
    "Totak+Vaamarvatn+Venemo": (
        ["Totak", "Vaamarvatn", "Venemo"],
        [],
        ["Songavatn+Bitdalsvatn", "Hyljelihyl"],
    ),
    "Langeidvatn": (["Langeidvatn"], [], []),
    "Vatjern": (["Vatjern"], [], ["Langeidvatn"]),
    "Vinjevatn": (["Vinjevatn"], [], ["Totak+Vaamarvatn+Venemo", "Vatjern"]),
    "Botnedalsvatn": (["Botnedalsvatn"], [], []),
    "Byrtevatn": (["Byrtevatn"], [], []),
    "Bandak": (["Bandak"], ["r_Vest_Vassdraget"], ["Vinjevatn", "Botnedalsvatn", "Byrtevatn"]),
}
INTAKE = {
    "Hogga": "Bandak",
    "Tokke": "Vinjevatn",
    "Lio": "Byrtevatn",
    "Byrte": "Botnedalsvatn",
    "Vinje": "Totak+Vaamarvatn+Venemo",
    "Haukeli": "Vatjern",
    "Songa": "Songavatn+Bitdalsvatn",
    "Kjela": "Foersvatn+Bordalsvatn",
    "Vesle Kjela": "Kjelavatn",
}

SPILL, POOL_V0, POOL_C = {}, {}, {}
for pool, (members, extra, ups) in POOLS.items():
    C = sum(MAXVOL[r] for r in members)
    V = sum(vol0(r) for r in members)
    inflow = sum(Q[r] for r in members + extra) * 0.0036 + sum(SPILL[u] for u in ups)
    POOL_V0[pool], POOL_C[pool] = V.copy(), C
    sp = np.zeros((N, 168))
    for t in range(168):
        V = V + inflow[:, t]
        s = np.maximum(V - C, 0.0)
        sp[:, t] = s
        V = V - s
    SPILL[pool] = sp

WATER = {}
for p in PLANTS:
    pool, qh = INTAKE[p], PLANT_QMAX[p] * 0.0036
    sp = SPILL[pool]
    rem = np.cumsum(sp[:, ::-1], axis=1)[:, ::-1]
    first = np.where((sp > 0).any(1), (sp > 0).argmax(1), 400)
    WATER[f"wb_forced_frac_{p}"] = np.repeat((sp.sum(1) / (qh * 168))[:, None], 168, 1)
    WATER[f"wb_headroom_hours_{p}"] = np.repeat(
        ((POOL_C[pool] - POOL_V0[pool]) / qh)[:, None], 168, 1
    )
    WATER[f"wb_hours_to_spill_{p}"] = first[:, None] - np.arange(168)[None, :]
    WATER[f"wb_forced_share_remaining_{p}"] = rem / qh / (168 - np.arange(168))[None, :]
for pool in ["Totak+Vaamarvatn+Venemo", "Foersvatn+Bordalsvatn", "Songavatn+Bitdalsvatn"]:
    WATER[f"wb_pool_fill_{pool}"] = np.repeat((POOL_V0[pool] / POOL_C[pool])[:, None], 168, 1)
for r in ["Vatjern", "Hyljelihyl", "Vinjevatn", "Langeidvatn"]:
    WATER[f"wb_inflow_h_{r}"] = Q[r]


In [ ]:
# -----------------------------
# Shared-river features
# -----------------------------
deficit = np.maximum(
    0,
    MF["minflow_h_r_Vest_Vassdraget"]
    - Q["r_Vest_Vassdraget"]
    - MF["minflow_h_b_Vinjevatn"]
    - MF["minflow_h_b_Byrtevatn"],
)
alt = np.minimum(cases["wv7_Venemo"], cases["wv7_Vinjevatn"]).to_numpy(np.float64)
byrt = cases["wv7_Byrtevatn"].to_numpy(np.float64)
RIVER = {
    "rv_deficit_h": deficit,
    "rv_deficit_over_lio_qmin": deficit / GEN["Lio_G1"]["q_min"],
    "rv_deficit_week_mean": np.repeat(deficit.mean(1)[:, None], 168, 1),
    "rv_alt_wv_minus_byrtevatn": np.repeat((alt - byrt)[:, None], 168, 1),
    "rv_alt_wv_over_byrtevatn": np.repeat((alt / np.maximum(byrt, 0.5))[:, None], 168, 1),
}


In [ ]:
# -----------------------------
# Weekly optimisation model
# -----------------------------
import numpy as np, scipy.sparse as sps
from scipy.optimize import linprog

K = 0.0036
MWH_PER_MM3 = 1e6 / 3600.0

PLANT_UP = {
    "Hogga": "Bandak",
    "Tokke": "Vinjevatn",
    "Lio": "Byrtevatn",
    "Byrte": "Botnedalsvatn",
    "Vinje": "Vaamarvatn",
    "Haukeli": "Vatjern",
    "Songa": "SONGA_IN",
    "Kjela": "Foersvatn",
    "Vesle Kjela": "Kjelavatn",
}
PLANT_DOWN = {
    "Hogga": "sea",
    "Tokke": "Bandak",
    "Lio": "rVV",
    "Byrte": "Byrtevatn",
    "Vinje": "Vinjevatn",
    "Haukeli": "Vinjevatn",
    "Songa": "Totak",
    "Kjela": "Hyljelihyl",
    "Vesle Kjela": "Foersvatn",
}

PATH = {
    "Bandak": ["Hogga"],
    "Vinjevatn": ["Tokke", "Hogga"],
    "Byrtevatn": ["Lio", "Hogga"],
    "Botnedalsvatn": ["Byrte", "Lio", "Hogga"],
    "Totak": ["Vinje", "Tokke", "Hogga"],
    "Vaamarvatn": ["Vinje", "Tokke", "Hogga"],
    "Venemo": ["Vinje", "Tokke", "Hogga"],
    "Hyljelihyl": ["Vinje", "Tokke", "Hogga"],
    "Songavatn": ["Songa", "Vinje", "Tokke", "Hogga"],
    "Bitdalsvatn": ["Songa", "Vinje", "Tokke", "Hogga"],
    "Langeidvatn": ["Haukeli", "Tokke", "Hogga"],
    "Vatjern": ["Haukeli", "Tokke", "Hogga"],
    "Foersvatn": ["Kjela", "Vinje", "Tokke", "Hogga"],
    "Bordalsvatn": ["Kjela", "Vinje", "Tokke", "Hogga"],
    "Langesae": ["Kjela", "Vinje", "Tokke", "Hogga"],
    "Kjelavatn": ["Vesle Kjela", "Kjela", "Vinje", "Tokke", "Hogga"],
    "Staavatn": ["Vesle Kjela", "Kjela", "Vinje", "Tokke", "Hogga"],
}

ARCS = [
    ("s_Bandak", "Bandak", "sea", 1000, "s"),
    ("b_Bandak", "Bandak", "sea", 10, "b"),
    ("b_Vinjevatn", "Vinjevatn", "rVV", 10, "b"),
    ("s_Vinjevatn", "Vinjevatn", "rVV", 1000, "s"),
    ("s_Byrtevatn", "Byrtevatn", "rVV", 1000, "s"),
    ("b_Byrtevatn", "Byrtevatn", "rVV", 10, "b"),
    ("s_Botnedalsvatn", "Botnedalsvatn", "rVV", 1000, "s"),
    ("b_Botnedalsvatn", "Botnedalsvatn", "rVV", 10, "b"),
    ("s_Kjelavatn", "Kjelavatn", "Hyljelihyl", 1000, "s"),
    ("b_Kjelavatn", "Kjelavatn", "Hyljelihyl", 10, "b"),
    ("s_Staavatn", "Staavatn", "Kjelavatn", 1000, "s"),
    ("r_Staavatn", "Staavatn", "Kjelavatn", None, "r"),
    ("s_Venemo", "Venemo", "Vinjevatn", 1000, "s"),
    ("b_Venemo", "Venemo", "Vinjevatn", 10, "b"),
    ("s_Langesae", "Langesae", "Foersvatn", 1000, "s"),
    ("r_Langesae", "Langesae", "Foersvatn", None, "r"),
    ("b_Hyljelihyl", "Hyljelihyl", "Vinjevatn", 10, "b"),
    ("r_Hyljelihyl", "Hyljelihyl", "Venemo", None, "r"),
    ("r_Langeidvatn", "Langeidvatn", "Vatjern", None, "r"),
    ("s_Langeidvatn", "Langeidvatn", "Vatjern", 1000, "s"),
    ("b_Bordalsvatn_Hyljelihyl", "Bordalsvatn", "Hyljelihyl", 10, "b"),
    ("b_Bordalsvatn_Venemo", "Bordalsvatn", "Venemo", 10, "b"),
    ("s_Bordalsvatn_Hyljelihyl", "Bordalsvatn", "Hyljelihyl", 1000, "s"),
    ("s_Bordalsvatn_Venemo", "Bordalsvatn", "Venemo", 1000, "s"),
    ("b_Foersvatn", "Foersvatn", "Hyljelihyl", 10, "b"),
    ("s_Foersvatn", "Foersvatn", "Hyljelihyl", 1000, "s"),
    ("b_Songavatn", "Songavatn", "Totak", 10, "b"),
    ("s_Songavatn", "Songavatn", "Totak", 1000, "s"),
    ("b_Bitdalsvatn", "Bitdalsvatn", "Totak", 10, "b"),
    ("s_Bitdalsvatn", "Bitdalsvatn", "Totak", 1000, "s"),
    ("b_Totak", "Totak", "rVV", 10, "b"),
    ("s_Totak", "Totak", "rVV", 1000, "s"),
    ("b_Vaamarvatn", "Vaamarvatn", "Vinjevatn", 10, "b"),
    ("s_Vaamarvatn", "Vaamarvatn", "Vinjevatn", 1000, "s"),
    ("b_Vatjern", "Vatjern", "Vinjevatn", 10, "b"),
    ("s_Vatjern", "Vatjern", "Vinjevatn", 1000, "s"),
    ("tun_Songavatn_Songa", "Songavatn", "SONGA_IN", 0, "t"),
    ("tun_Bitdalsvatn_Songa", "Bitdalsvatn", "SONGA_IN", 0, "t"),
]
FREE_TUNNELS = [("Totak", "Vaamarvatn"), ("Venemo", "Totak"), ("Bordalsvatn", "Foersvatn")]
RIVER_SEGMENTS = [(40, 1.6), (60, 2.9), (900, 3.25)]
MINFLOW_ARCS = ["b_Kjelavatn", "b_Hyljelihyl", "b_Vinjevatn", "b_Byrtevatn", "b_Bandak"]
MINVOL_RES = ["Totak", "Byrtevatn", "Staavatn"]
PEN_FLOW, PEN_VOL = 1e5, 1e8


class WeeklyLP:
    def __init__(
        self, reservoirs, maxvol, units, unit_plant, gen, T=168, value_mode="release", qbest=None
    ):
        self.T, self.R, self.maxvol = T, list(reservoirs), maxvol
        self.qbest = qbest or {}
        self.plants = list(dict.fromkeys(unit_plant[u] for u in units))
        self.nunit = {p: sum(unit_plant[u] == p for u in units) for p in self.plants}
        g = {p: gen[next(u for u in units if unit_plant[u] == p)] for p in self.plants}
        self.g = g
        self.eta = {p: g[p]["p_max"] / g[p]["q_max"] for p in self.plants}
        E = {r: MWH_PER_MM3 * sum(self.eta[p] for p in PATH[r]) for r in self.R}
        Eloc = {r: MWH_PER_MM3 * self.eta[PATH[r][0]] for r in self.R}
        self.mode = value_mode
        self.E = {"cumulative": E, "local": Eloc, "release": {r: 0.0 for r in self.R}}[value_mode]

        self.own_eta = {r: self.eta[PATH[r][0]] for r in self.R}
        self._build()

    def _build(self):
        T = self.T
        self.cols = {}
        n = 0

        def add(name, length):
            nonlocal n
            self.cols[name] = np.arange(n, n + length)
            n += length

        for p in self.plants:
            for v in ("U", "Q", "Q1", "Q2", "S", "D"):
                add(f"{v}_{p}", T)
        for a in ARCS:
            if a[4] == "r":
                for i in range(len(RIVER_SEGMENTS)):
                    add(f"{a[0]}#{i}", T)
            else:
                add(a[0], T)
        for fr, to in FREE_TUNNELS:
            add(f"tun_{fr}_{to}", T)
        add("F_rVV", T)
        for r in self.R:
            add(f"V_{r}", T + 1)
        for a in MINFLOW_ARCS:
            add(f"slk_{a}", T)
        add("slk_rVV", T)
        for r in MINVOL_RES:
            add(f"slkv_{r}", T)
        self.n = n

        # Reservoir balances and river-node constraints.
        Ei, Ej, Ev = [], [], []
        self.eq_rows = {}
        m = 0

        def eq_row(name):
            nonlocal m
            self.eq_rows[name] = np.arange(m, m + T)
            m += T

        t = np.arange(T)
        out_of, into = {}, {}

        def flow(node_from, node_to, colname, coef=1.0):
            out_of.setdefault(node_from, []).append((colname, coef))
            into.setdefault(node_to, []).append((colname, coef))

        for p in self.plants:
            flow(PLANT_UP[p], PLANT_DOWN[p], f"Q_{p}")
        for a in ARCS:
            if a[4] == "r":
                for i in range(len(RIVER_SEGMENTS)):
                    flow(a[1], a[2], f"{a[0]}#{i}")
            else:
                flow(a[1], a[2], a[0])
        for fr, to in FREE_TUNNELS:
            flow(fr, to, f"tun_{fr}_{to}")
        flow("rVV", "Bandak", "F_rVV")
        for r in self.R:
            eq_row(f"bal_{r}")
            rows = self.eq_rows[f"bal_{r}"]
            V = self.cols[f"V_{r}"]
            Ei += [rows, rows]
            Ej += [V[1:], V[:-1]]
            Ev += [np.ones(T), -np.ones(T)]
            for c, coef in out_of.get(r, []):
                Ei.append(rows)
                Ej.append(self.cols[c])
                Ev.append(np.full(T, K * coef))
            for c, coef in into.get(r, []):
                Ei.append(rows)
                Ej.append(self.cols[c])
                Ev.append(np.full(T, -K * coef))
        for node in ("rVV", "SONGA_IN"):
            eq_row(f"node_{node}")
            rows = self.eq_rows[f"node_{node}"]
            for c, coef in out_of.get(node, []):
                Ei.append(rows)
                Ej.append(self.cols[c])
                Ev.append(np.full(T, coef))
            for c, coef in into.get(node, []):
                Ei.append(rows)
                Ej.append(self.cols[c])
                Ev.append(np.full(T, -coef))
        for p in self.plants:
            eq_row(f"qdef_{p}")
            rows = self.eq_rows[f"qdef_{p}"]
            Ei += [rows] * 4
            Ej += [
                self.cols[f"Q_{p}"],
                self.cols[f"U_{p}"],
                self.cols[f"Q1_{p}"],
                self.cols[f"Q2_{p}"],
            ]
            Ev += [np.ones(T), -np.full(T, self.g[p]["q_min"]), -np.ones(T), -np.ones(T)]
        self.m_eq = m
        self.A_eq = sps.csr_matrix(
            (np.concatenate(Ev), (np.concatenate(Ei), np.concatenate(Ej))), shape=(m, n)
        )

        # Capacity, start/stop and minimum-flow constraints.
        Ii, Ij, Iv = [], [], []
        self.ub_rows = {}
        k = 0

        def ub_row(name, length=T):
            nonlocal k
            self.ub_rows[name] = np.arange(k, k + length)
            k += length

        for p in self.plants:
            g = self.g[p]
            U, Q, S, D = (self.cols[f"{v}_{p}"] for v in "UQSD")
            qb = self._qb(p)[0]
            Q1, Q2 = self.cols[f"Q1_{p}"], self.cols[f"Q2_{p}"]
            ub_row(f"q1_{p}")
            r_ = self.ub_rows[f"q1_{p}"]
            Ii += [r_, r_]
            Ij += [Q1, U]
            Iv += [np.ones(T), -np.full(T, qb - g["q_min"])]
            ub_row(f"q2_{p}")
            r_ = self.ub_rows[f"q2_{p}"]
            Ii += [r_, r_]
            Ij += [Q2, U]
            Iv += [np.ones(T), -np.full(T, g["q_max"] - qb)]
            ub_row(f"start_{p}", T - 1)
            r_ = self.ub_rows[f"start_{p}"]
            Ii += [r_, r_, r_]
            Ij += [U[1:], U[:-1], S[1:]]
            Iv += [np.ones(T - 1), -np.ones(T - 1), -np.ones(T - 1)]
            ub_row(f"stop_{p}", T - 1)
            r_ = self.ub_rows[f"stop_{p}"]
            Ii += [r_, r_, r_]
            Ij += [U[:-1], U[1:], D[1:]]
            Iv += [np.ones(T - 1), -np.ones(T - 1), -np.ones(T - 1)]
        for a in MINFLOW_ARCS:
            ub_row(f"mf_{a}")
            r_ = self.ub_rows[f"mf_{a}"]
            Ii += [r_, r_]
            Ij += [self.cols[a], self.cols[f"slk_{a}"]]
            Iv += [-np.ones(T), -np.ones(T)]
        ub_row("mf_rVV")
        r_ = self.ub_rows["mf_rVV"]
        Ii += [r_, r_]
        Ij += [self.cols["F_rVV"], self.cols["slk_rVV"]]
        Iv += [-np.ones(T), -np.ones(T)]
        for r in MINVOL_RES:
            ub_row(f"mv_{r}")
            r_ = self.ub_rows[f"mv_{r}"]
            Ii += [r_, r_]
            Ij += [self.cols[f"V_{r}"][1:], self.cols[f"slkv_{r}"]]
            Iv += [-np.ones(T), -np.ones(T)]
        self.m_ub = k
        self.A_ub = sps.csr_matrix(
            (np.concatenate(Iv), (np.concatenate(Ii), np.concatenate(Ij))), shape=(k, n)
        )

        # Objective coefficients and variable bounds.
        c = np.zeros(n)
        lb = np.zeros(n)
        ub = np.full(n, np.inf)
        for p in self.plants:
            g = self.g[p]
            ub[self.cols[f"U_{p}"]] = self.nunit[p]
            c[self.cols[f"S_{p}"]] = g["startcost"]
            c[self.cols[f"D_{p}"]] = g["stopcost"]
        for a in ARCS:
            if a[4] == "r":
                for i, (w, cst) in enumerate(RIVER_SEGMENTS):
                    c[self.cols[f"{a[0]}#{i}"]] = cst
                    ub[self.cols[f"{a[0]}#{i}"]] = w
            else:
                c[self.cols[a[0]]] = a[3]
                ub[self.cols[a[0]]] = 1000 if a[4] in "bs" else np.inf
        for fr, to in FREE_TUNNELS:
            lb[self.cols[f"tun_{fr}_{to}"]] = -1000
            ub[self.cols[f"tun_{fr}_{to}"]] = 1000
        for r in self.R:
            ub[self.cols[f"V_{r}"][1:]] = self.maxvol[r]
        for a in MINFLOW_ARCS:
            c[self.cols[f"slk_{a}"]] = PEN_FLOW
        c[self.cols["slk_rVV"]] = PEN_FLOW
        for r in MINVOL_RES:
            c[self.cols[f"slkv_{r}"]] = PEN_VOL
        self.c0, self.lb0, self.ub0 = c, lb, ub

    def _qb(self, p):
        """Best-efficiency point (q, p) per unit; falls back to the straight line if missing or not concave."""
        g = self.g[p]
        mid = ((g["q_min"] + g["q_max"]) / 2, (g["p_min"] + g["p_max"]) / 2)
        if p not in self.qbest:
            return mid
        qb, pb = self.qbest[p]
        ok = g["q_min"] < qb < g["q_max"] and (pb - g["p_min"]) / (qb - g["q_min"]) >= (
            g["p_max"] - pb
        ) / (g["q_max"] - qb)
        return (qb, pb) if ok else mid

    def skip_cost(self, src, dst, wv_end):
        dpath = [] if dst == "sea" else PATH["Bandak" if dst == "rVV" else dst]
        cost = 0.0
        for p in PATH[src]:
            if p in dpath:
                break
            up = "Songavatn" if p == "Songa" else PLANT_UP[p]
            cost += wv_end[up] * self.eta[p]
        return cost

    def solve(self, price, inflow, v0, wv_end, minflow, minvol):
        arrays = [
            price,
            *inflow.values(),
            *minflow.values(),
            *minvol.values(),
            np.array(list(v0.values())),
            np.array(list(wv_end.values())),
        ]
        if not all(np.isfinite(np.asarray(a, float)).all() for a in arrays):
            self.last_status = ("nan input",)
            return None
        out = self._solve(price, inflow, v0, wv_end, minflow, minvol, drain_hours=24)
        if out is None:
            out = self._solve(price, inflow, v0, wv_end, minflow, minvol, drain_hours=None)
        return out

    def _solve(self, price, inflow, v0, wv_end, minflow, minvol, drain_hours):
        T = self.T
        c = self.c0.copy()
        lb = self.lb0.copy()
        ub = self.ub0.copy()
        for p in self.plants:
            g = self.g[p]
            qb, pb = self._qb(p)
            s1 = (pb - g["p_min"]) / (qb - g["q_min"])
            s2 = (g["p_max"] - pb) / (g["q_max"] - qb)
            c[self.cols[f"U_{p}"]] -= price * g["p_min"]
            c[self.cols[f"Q1_{p}"]] -= price * s1
            c[self.cols[f"Q2_{p}"]] -= price * s2
        if self.mode == "release":
            for p in self.plants:
                r = "Songavatn" if p == "Songa" else PLANT_UP[p]
                c[self.cols[f"Q_{p}"]] += wv_end[r] * self.eta[p]
            for a in ARCS:
                if a[4] == "t":
                    continue
                cost = self.skip_cost(a[1], a[2], wv_end)
                cols = (
                    [self.cols[f"{a[0]}#{i}"] for i in range(len(RIVER_SEGMENTS))]
                    if a[4] == "r"
                    else [self.cols[a[0]]]
                )
                for cc in cols:
                    c[cc] += cost
        for r in self.R:
            V = self.cols[f"V_{r}"]
            lb[V[0]] = ub[V[0]] = v0[r]
            excess = max(0.0, v0[r] - self.maxvol[r])
            ub[V[1:]] = self.maxvol[r] + (
                excess * np.clip(1 - np.arange(1, T + 1) / drain_hours, 0, None)
                if drain_hours
                else excess
            )
            c[V[-1]] -= wv_end[r] * self.E[r]
        b_eq = np.zeros(self.m_eq)
        for r in self.R:
            b_eq[self.eq_rows[f"bal_{r}"]] = K * inflow[r]
        b_eq[self.eq_rows["node_rVV"]] = inflow["r_Vest_Vassdraget"]
        b_ub = np.zeros(self.m_ub)
        for a in MINFLOW_ARCS:
            b_ub[self.ub_rows[f"mf_{a}"]] = -minflow.get(a, np.zeros(T))
        b_ub[self.ub_rows["mf_rVV"]] = -minflow.get("r_Vest_Vassdraget", np.zeros(T))
        for r in MINVOL_RES:
            b_ub[self.ub_rows[f"mv_{r}"]] = -minvol.get(r, np.zeros(T))
        # Solve the relaxed weekly scheduling problem.
        res = linprog(
            c,
            A_ub=self.A_ub,
            b_ub=b_ub,
            A_eq=self.A_eq,
            b_eq=b_eq,
            bounds=np.column_stack([lb, ub]),
            method="highs",
        )
        if res.status != 0:
            self.last_status = (res.status, res.message)
            return None
        x = res.x
        w = -res.eqlin.marginals

        def wnode(node):
            if node == "sea":
                return np.zeros(T)
            if node in ("rVV", "SONGA_IN"):
                return w[self.eq_rows[f"node_{node}"]] / K
            return w[self.eq_rows[f"bal_{node}"]]

        out = {}
        for p in self.plants:
            U, Q = x[self.cols[f"U_{p}"]], x[self.cols[f"Q_{p}"]]
            out[f"U_{p}"] = U
            out[f"qfrac_{p}"] = Q / (self.g[p]["q_max"] * self.nunit[p])
            wv_rel = (
                wv_end["Songavatn" if p == "Songa" else PLANT_UP[p]]
                if self.mode == "release"
                else 0.0
            )
            out[f"margin_{p}"] = (
                price - wv_rel - K * (wnode(PLANT_UP[p]) - wnode(PLANT_DOWN[p])) / self.eta[p]
            )
            out[f"wup_{p}"] = K * wnode(PLANT_UP[p]) / self.eta[p]
            out[f"wdn_{p}"] = K * wnode(PLANT_DOWN[p]) / self.eta[p]
        for r in self.R:
            out[f"Vfrac_{r}"] = x[self.cols[f"V_{r}"][1:]] / self.maxvol[r]
        out["release_nonplant"] = sum(x[self.cols[a[0]]] for a in ARCS if a[4] in "bs")
        out["obj"] = res.fun
        return out


In [ ]:
# -----------------------------
# Prepare optimisation inputs
# -----------------------------
import multiprocessing as mp
from concurrent.futures import ProcessPoolExecutor

LP_RES = list(MAXVOL)
GEN_LP = {
    u: dict(
        GEN[u],
        startcost=topo["model"]["generator"][u]["startcost_const"],
        stopcost=topo["model"]["generator"][u]["stopcost_const"],
    )
    for u in UNITS
}
QBEST = {}
for u in UNITS:
    p = UNIT_PLANT[u]
    if p in QBEST:
        continue
    cs = topo["model"]["generator"][u]["turb_eff_curves"]
    cs = cs if isinstance(cs, list) else [cs]
    cv = cs[len(cs) // 2]
    qx, ef = np.array(cv["x"]), np.array(cv["y"])
    pw = qx * ef
    ib = int(np.argmax(ef))
    QBEST[p] = (float(qx[ib]), float(GEN[u]["p_max"] * pw[ib] / pw[-1]))


wvraw = pd.read_csv(WV_FILE)
wvraw["date"] = pd.to_datetime(wvraw["date"], utc=True)
wvraw = wvraw.set_index("date").sort_index()
WV_END = wvraw.reindex(cases["starttime"] + pd.Timedelta(days=7))[LP_RES]
assert WV_END.notna().all().all(), "terminal water value missing for some cases"
WV_END = WV_END.to_numpy(np.float64)
for r in ["Bandak", "Vinjevatn", "Byrtevatn", "Kjelavatn"]:
    assert np.allclose(
        WV_END[:, LP_RES.index(r)], cases[f"wv7_{r}"].to_numpy(), atol=1e-6
    ), f"WV_END misaligned for {r}"
V0 = np.column_stack([vol0(r) for r in LP_RES])
H = lambda c: FEAT[c].to_numpy(np.float64).reshape(N, 168)
LP_MINFLOW = {a: H(f"minflow_h_{a}") for a in MINFLOW_ARCS}
LP_MINFLOW["r_Vest_Vassdraget"] = H("minflow_h_r_Vest_Vassdraget")
LP_MINVOL = {r: H(f"minvol_h_{r}") for r in MINVOL_RES}
LP_Q = {c: Q[c] for c in Q}

_LP = None


def _solve_case(i):
    """Solve the weekly LP for case i (one LP object per worker process)."""
    global _LP
    if _LP is None:
        _LP = WeeklyLP(LP_RES, MAXVOL, UNITS, UNIT_PLANT, GEN_LP, value_mode="release", qbest=QBEST)
    out = _LP.solve(
        PRICE[i],
        {c: LP_Q[c][i] for c in LP_Q},
        dict(zip(LP_RES, V0[i])),
        dict(zip(LP_RES, WV_END[i])),
        {a: LP_MINFLOW[a][i] for a in LP_MINFLOW},
        {r: LP_MINVOL[r][i] for r in LP_MINVOL},
    )
    return i, out


In [ ]:
# -----------------------------
# Solve the weekly cases
# -----------------------------
t0 = time.time()
workers = LP_WORKERS or os.cpu_count()
with ProcessPoolExecutor(max_workers=workers, mp_context=mp.get_context("fork")) as ex:
    LP_OUT = dict(ex.map(_solve_case, range(N), chunksize=16))
failed = [i for i, o in LP_OUT.items() if o is None]
print(f"LP solved {N - len(failed)}/{N} cases with {workers} workers in {time.time() - t0:.0f}s")
assert len(failed) <= 0.01 * N, "too many LP failures"


In [ ]:
# -----------------------------
# Optimisation features
# -----------------------------
ORDER = {
    "Tokke": ["Tokke_G1", "Tokke_G3", "Tokke_G4", "Tokke_G2"],
    "Vinje": ["Vinje_G1", "Vinje_G2", "Vinje_G3"],
}
INTAKE_RES = {p: ("Songavatn" if p == "Songa" else PLANT_UP[p]) for p in PLANTS}


def lp_arr(key):
    a = np.full((N, 168), np.nan)
    for i, o in LP_OUT.items():
        if o is not None:
            a[i] = o[key]
    return a


LPF = {}
for p in PLANTS:
    U = lp_arr(f"U_{p}")
    LPF[f"lp_U_{p}"] = U
    LPF[f"lp_qfrac_{p}"] = lp_arr(f"qfrac_{p}")
    LPF[f"lp_margin_{p}"] = lp_arr(f"margin_{p}")
    LPF[f"lp_Vfrac_{p}"] = lp_arr(f"Vfrac_{INTAKE_RES[p]}")
    LPF[f"lp_wup_{p}"] = lp_arr(f"wup_{p}")
    LPF[f"lp_wdn_{p}"] = lp_arr(f"wdn_{p}")
for u in UNITS:
    p = UNIT_PLANT[u]
    rank = ORDER[p].index(u) if p in ORDER else 0
    LPF[f"lp_unit_{u}"] = np.clip(LPF[f"lp_U_{p}"] - rank, 0, 1)
LPF["lp_release_nonplant"] = lp_arr("release_nonplant")
print(len(LPF), "LP features")


In [ ]:
# -----------------------------
# Final feature matrix
# -----------------------------
B1 = pd.DataFrame(
    {k: np.asarray(v, np.float32).ravel() for k, v in {**WATER, **RIVER, **LPF}.items()}
)
FEAT = pd.concat([FEAT, B1], axis=1)
assert not np.isinf(FEAT.to_numpy(np.float32)).any()
HK_KEYS = ("Haukeli", "Vatjern", "Langeidvatn")
WATER_NOHK = [c for c in WATER if not any(k in c for k in HK_KEYS)]
BASE = CONTROL + list(RIVER) + WATER_NOHK
BASE_L = BASE + list(LPF)
del WATER, RIVER, B1, LP_OUT
lab_rows, test_rows = np.repeat(LAB, 168), np.repeat(~LAB, 168)
assert (
    FEAT.loc[test_rows, list(LPF)].notna().mean().min() > 0.99
), "LP features missing for test cases"
print("features:", len(BASE_L), "| labelled rows", lab_rows.sum(), "| test rows", test_rows.sum())


In [ ]:
# -----------------------------
# Device check
# -----------------------------
try:
    import cupy as cp
except ImportError:
    cp = None


def to_dev(a):
    return cp.asarray(a) if (cp is not None and DEVICE == "cuda") else a


def to_np(a):
    return a.get() if hasattr(a, "get") else np.asarray(a)


if DEVICE == "cuda":
    print(
        subprocess.run(
            ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"],
            capture_output=True,
            text=True,
        ).stdout
    )
    assert xgb.build_info().get("USE_CUDA", False), "This xgboost build has no CUDA support"
    _X = np.random.rand(5000, 10).astype(np.float32)
    _y = (_X[:, 0] > 0.5).astype(int)
    _m = xgb.XGBClassifier(n_estimators=5, tree_method="hist", device="cuda").fit(
        to_dev(_X), to_dev(_y)
    )
    _dev = json.loads(_m.get_booster().save_config())["learner"]["generic_param"]["device"]
    assert _dev.startswith("cuda"), f"Booster reports device={_dev}"
    print("CUDA verified: booster device =", _dev)


def logit(p):
    p = np.clip(p, 1e-6, 1 - 1e-6)
    return np.log(p / (1 - p))


def sigm(z):
    return 1 / (1 + np.exp(-z))


In [ ]:
# -----------------------------
# Train one model per generating unit
# -----------------------------
def fit_predict(Xtr, ytr, Xte, seed):
    m = xgb.XGBClassifier(**BASE_PARAMS, random_state=seed, device=DEVICE)
    m.fit(Xtr, ytr, verbose=False)
    return to_np(m.predict_proba(Xte))[:, 1]


t0 = time.time()
X = FEAT[BASE_L].to_numpy(np.float32)
Xtr, Xte = to_dev(X[lab_rows]), to_dev(X[test_rows])
del X
PRED_L = {}
for seed in SEEDS:
    PRED_L[seed] = np.column_stack(
        [
            fit_predict(Xtr, to_dev(Y[lab_rows, j].astype(np.int32)), Xte, seed)
            for j in range(len(UNITS))
        ]
    )
    print(f"L seed {seed} done ({time.time() - t0:.0f}s)")
del Xtr, Xte


In [ ]:
# -----------------------------
# Pooled model helpers
# -----------------------------
OWN_TEMPLATES = [
    "price_minus_wvup_{p}",
    "fe_price_minus_wvnet_{p}",
    "fe_above_wv_block_hours_{p}",
    "fe_remaining_above_wv_fraction_{p}",
    "fe_remaining_positive_spread_mean_{p}",
    "fe_relative_price_spread_{p}",
    "wvup_{p}",
    "wvnet_{p}",
    "spread_{p}",
    "frac_price_gt_wv_{p}",
    "fe_initial_fill_up_{p}",
    "wb_forced_frac_{p}",
    "wb_headroom_hours_{p}",
    "wb_hours_to_spill_{p}",
    "wb_forced_share_remaining_{p}",
]
OWN_LP_TEMPLATES = [
    "lp_unit_{u}",
    "lp_U_{p}",
    "lp_qfrac_{p}",
    "lp_margin_{p}",
    "lp_wup_{p}",
    "lp_wdn_{p}",
    "lp_Vfrac_{p}",
]
UNIT_CONST = {
    u: [
        GEN[u]["p_max"],
        GEN[u]["q_max"],
        GEN[u]["sc"],
        int(u.split("_G")[-1]),
        sum(UNIT_PLANT[v] == UNIT_PLANT[u] for v in UNITS),
    ]
    for u in UNITS
}


def own_block(unit, rows):
    """[unit_id | own-plant columns | unit constants] for one unit and the selected rows."""
    p = UNIT_PLANT[unit]
    out = []
    for tpl in OWN_TEMPLATES + OWN_LP_TEMPLATES:
        c = tpl.format(p=p, u=unit)
        if c in FEAT and not (p == "Haukeli" and c.startswith("wb_")):
            out.append(FEAT[c].to_numpy(np.float32)[rows])
        else:
            assert p == "Haukeli" and (
                c.startswith("wb_") or c.startswith("fe_initial_fill_up")
            ), f"missing own feature {c}"
            out.append(np.full(rows.sum(), np.nan, np.float32))
    const = np.tile(np.array(UNIT_CONST[unit], np.float32), (rows.sum(), 1))
    uid = np.full((rows.sum(), 1), UNITS.index(unit), np.float32)
    return np.hstack([uid, np.column_stack(out), const])


N_OWN = 1 + len(OWN_TEMPLATES) + len(OWN_LP_TEMPLATES) + 5
FTYPES = ["q"] * len(BASE_L) + ["c"] + ["q"] * (N_OWN - 1)


class StackIter(xgb.DataIter):
    """Feeds the stacked (rows × unit) training matrix to XGBoost one unit at a time."""

    def __init__(self, Xbase, rows):
        self.Xb, self.rows, self.i = Xbase, rows, 0
        super().__init__()

    def next(self, input_data):
        if self.i == len(UNITS):
            return False
        X = np.hstack([self.Xb, own_block(UNITS[self.i], self.rows)])
        input_data(
            data=to_dev(X),
            label=to_dev(Y[self.rows, self.i].astype(np.float32)),
            feature_types=FTYPES,
        )
        self.i += 1
        return True

    def reset(self):
        self.i = 0


def pooled_params(seed):
    p = {k: v for k, v in BASE_PARAMS.items() if k not in ("n_estimators", "n_jobs")}
    p.update(device=DEVICE, seed=seed, max_cat_to_onehot=1)
    return p


def pooled_predict(bst, rows):
    Xb = FEAT[BASE_L].to_numpy(np.float32)[rows]
    out = np.zeros((rows.sum(), len(UNITS)), np.float32)
    for j, u in enumerate(UNITS):
        d = xgb.DMatrix(
            to_dev(np.hstack([Xb, own_block(u, rows)])),
            feature_types=FTYPES,
            enable_categorical=True,
        )
        out[:, j] = to_np(bst.predict(d))
    return out


In [ ]:
# -----------------------------
# Train the pooled model
# -----------------------------
t0 = time.time()
dtr = xgb.QuantileDMatrix(
    StackIter(FEAT[BASE_L].to_numpy(np.float32)[lab_rows], lab_rows),
    max_bin=256,
    enable_categorical=True,
)
PRED_PL = {}
for seed in SEEDS:
    PRED_PL[seed] = pooled_predict(xgb.train(pooled_params(seed), dtr, PL_ROUNDS), test_rows)
    print(f"PL seed {seed} done ({time.time() - t0:.0f}s)")
del dtr


In [ ]:
# -----------------------------
# Blend predictions and save CSV
# -----------------------------
P = np.mean([sigm(0.5 * logit(PRED_PL[s]) + 0.5 * logit(PRED_L[s])) for s in SEEDS], 0)

sample = pd.read_csv(f"{SP}/sample_submission.csv")
runs = RUN[test_rows].reshape(-1, 168)[:, 0]
Pc = P.reshape(-1, 168, len(UNITS))
assert (TT[test_rows].reshape(-1, 168) == np.arange(168)).all()
wide = {"Run No": runs}
for j, u in enumerate(UNITS):
    for t in range(168):
        wide[f"result_committed_{u}_t{t}"] = Pc[:, t, j]
sub = pd.DataFrame(wide).set_index("Run No").loc[sample["Run No"]].reset_index()[sample.columns]
assert sub.shape == sample.shape == (725, 2353) and list(sub.columns) == list(sample.columns)
assert (sub["Run No"].to_numpy() == sample["Run No"].to_numpy()).all()
v = sub.iloc[:, 1:].to_numpy()
assert np.isfinite(v).all() and v.min() >= 0 and v.max() <= 1

path = OUTPUT_DIR / OUT_NAME
sub.to_csv(path, index=False)
print(
    f"wrote {path} {sub.shape} | mean prediction {v.mean():.4f} (scored run: 0.4680) | total time {(time.time() - T_START) / 60:.1f} min"
)
